# Suite CHK — Checking an architecture

`Interactions.Check()(root)` lists the problems of the architecture reachable from `root` that no `unique` clause
expresses, Structure's included, each a line naming the object by its schema and dotted path. mbse-schemas'
`Validators.Validate` reports the rest (scalar types, `unique` clauses, which schemas fill a link). These cases make
each problem once, and the seams where Structure and Interactions meet.

In [ ]:
import subprocess
import sys

from mbse.Schemas.Framework import Builders, Validators
from mbse.Architecture.Systems.Examples import Pipeline
from mbse.Architecture.Systems.Framework import Interactions, Structure


def pipeline():
    """A store holding the example pipeline, and its parts."""
    store = Builders.Store()
    return store, Pipeline.build(store)


def check(p):
    """The problems Interactions.Check reports from the pipeline module."""
    return Interactions.Check()(p.pipeline)

## CHK-01 · The example pipeline has no problems, and checks itself as a script

In [ ]:
result = subprocess.run([sys.executable, "-m", "mbse.Architecture.Systems.Examples.Pipeline"], capture_output=True,
                        text=True, timeout=300)
assert result.returncode == 0 and result.stdout == "Pipeline: all checks passed\n", result.stderr

store, p = pipeline()
assert check(p) == []
assert Validators.Validate(store).Reachable(p.schemas.Module, p.pipeline) == []
assert Interactions.Check()(p.impl) == [] and Interactions.Check()(p.sections["processing"]) == []
assert Interactions.Check().label(p.dataflow) == "Structure.Axis 'dataflow'"  # what has no path: as Structure names it
assert Interactions.Check().label(p.instance["sensor"]) == "Interactions.Instance 'reference.sensor'"

## CHK-02 · Ports have a direction; implementations and instances are of a module; an instance has one name

In [ ]:
store, p = pipeline()
with store.transaction():
    store.Interactions.Module(p.logger, lambda m: m.interfaces(lambda a: a.insert(lambda e: e.name("debug").intf(
        lambda i: i.ports(lambda a: (
            a
            .insert(lambda e: e.name("plain").port(lambda q: q))
            .insert(lambda e: e.name("both").port(lambda q: q.direction("inout")))
        ))))))
    store.Interactions.Implementation(lambda x: x.name("draft"))  # implements nothing
    store.Interactions.Implementation(p.impl, lambda x: x.instances(lambda a: a.insert(lambda e: e.name("spare").instance(
        lambda i: i.name("extra")))))  # of no module, and named otherwise as a container
    store.Interactions.Instance(lambda i: i.name("orphan").module(lambda a: a.insert(lambda e: e.module(p.logger))))
problems = check(p)
assert problems == [
    "Interactions.Instance 'reference.spare' is of no module",
    "Interactions.Instance 'reference.spare' is named 'extra' as a container, not 'spare'",
    "Interactions.Instance 'orphan' is in no implementation",  # no path: named as Structure names it
    "Interactions.Port 'Logger.debug.plain' has no direction: give it 'in' or 'out'",
    "Interactions.Port 'Logger.debug.both' has no direction: give it 'in' or 'out'",
], problems  # the implementation "draft" implements nothing, so nothing reached from the pipeline reaches it

# Reached from its own side, an implementation of nothing says so.
with store.transaction():
    draft_impl = store.Interactions.Implementation(lambda x: x.name("draft"))
assert Interactions.Check()(draft_impl) == ["Interactions.Implementation 'draft' implements no module"]
with store.transaction():
    nameless = store.Interactions.Implementation(lambda x: x)
assert Interactions.Check()(nameless) == ["Interactions.Implementation #0 implements no module"]

## CHK-03 · Channels: in an implementation, of a schema, distributed as one of the three, attached within

In [ ]:
store, p = pipeline()
library = store.singleton("Interactions.Library")
with store.transaction():
    other = store.Interactions.Implementation(lambda x: x.name("other").module(lambda a: a.insert(
        lambda e: e.module(p.pipeline))).instances(lambda a: a.insert(lambda e: e.name("stranger").instance(
            lambda i: i.module(lambda a: a.insert(lambda e: e.module(p.sensor)))))))
    stranger = other.instances.at("stranger").instance
    store.Interactions.Implementation(p.impl, lambda x: x.channels(lambda a: (
        a
        .insert(lambda e: e.name("odd").channel(lambda c: c.distribution("MULTICAST")))
        .insert(lambda e: e.name("sinks").channel(lambda c: c.message(lambda a: a.insert(lambda e: e.schema(p.Sample))))))))
    odd, sinks = p.impl.channels.at("odd").channel, p.impl.channels.at("sinks").channel
    samples = p.sensor.interfaces.at("samples").intf
    records = p.logger.interfaces.at("records").intf
    # attached to an instance of another implementation, through an interface of another module, by a port of another
    store.Interactions.Channel(odd, lambda c: c.connections(lambda a: (
        a
        .insert(lambda e: e.instance(stranger).intf(samples).port(samples.ports.at("tx").port))
        .insert(lambda e: e.instance(p.instance["filter"]).intf(records).port(samples.ports.at("tx").port))
    )))
    Pipeline.connect(store, sinks, p.instance["logger"], "records", "rx")  # destinations only
    lost = store.Interactions.Channel(lambda c: c.message(lambda a: a.insert(lambda e: e.schema(p.Sample))))
    Pipeline.connect(store, lost, p.instance["sensor"], "samples", "tx")  # a source only, in no implementation
problems = check(p)
assert problems == [
    "Interactions.Channel 'reference.odd' carries messages of no schema",
    "Interactions.Channel 'reference.odd' distributes 'MULTICAST', not one of BROADCAST, ANYCAST, ROUND_ROBIN",
    "Interactions.Channel 'reference.odd' is attached to Interactions.Instance 'other.stranger', outside its "
    "implementation",
    "Interactions.Channel 'reference.odd' is attached to Interactions.Interface 'Logger.records', not an interface of "
    "Interactions.Module 'Filter'",
    "Interactions.Channel 'reference.odd' is attached through Interactions.Port 'Sensor.samples.tx', not a port of "
    "Interactions.Interface 'Logger.records'",
    "Interactions.Channel 'reference.odd' has no destination",
    "Interactions.Channel 'reference.sinks' has no source",
    "Interactions.Channel #27 is in no implementation",
    "Interactions.Channel #27 has no destination",
], problems

# A channel attached nowhere has neither end; one attached through a port without a direction is the port's problem.
with store.transaction():
    store.Interactions.Implementation(p.impl, lambda x: x.channels(lambda a: a.insert(lambda e: e.name("idle").channel(
        lambda c: c.message(lambda a: a.insert(lambda e: e.schema(p.Sample)))))))
idle = [x for x in check(p) if "'reference.idle'" in x]
assert idle == ["Interactions.Channel 'reference.idle' has no source",
                "Interactions.Channel 'reference.idle' has no destination"]
with store.transaction():
    store.Interactions.Module(p.logger, lambda m: m.interfaces(lambda a: a.insert(lambda e: e.name("tap").intf(
        lambda i: i.ports(lambda a: a.insert(lambda e: e.name("any").port(lambda q: q)))))))
    Pipeline.connect(store, p.impl.channels.at("idle").channel, p.instance["logger"], "tap", "any")
problems = check(p)
assert [x for x in problems if "'reference.idle'" in x] == []
assert "Interactions.Port 'Logger.tap.any' has no direction: give it 'in' or 'out'" in problems

## CHK-04 · Seams with Structure: an implementation holds its instances and channels hard

In [ ]:
store, p = pipeline()
# The sections group what the implementation holds: its instances and channels, which no Contains entry holds hard.
assert all(Structure.hard_containers(x) == [] for x in (*p.instance.values(), *p.channel.values()))
assert all(Interactions.Check().hard_containers(x)[0] is p.impl for x in (*p.instance.values(), *p.channel.values()))

with store.transaction():
    other = store.Interactions.Implementation(lambda x: x.name("other").module(lambda a: a.insert(
        lambda e: e.module(p.pipeline))).instances(lambda a: a.insert(lambda e: e.name("spare").instance(
            lambda i: i.module(lambda a: a.insert(lambda e: e.module(p.sensor)))))))
    spare = other.instances.at("spare").instance
    # a section of one implementation grouping an instance of another
    store.Structure.Container(p.sections["acquisition"], lambda c: c.contents(lambda a: a.insert(lambda e: e.c2(spare))))
    # an instance held hard by a container as well as by its implementation
    store.Structure.Container(lambda c: c.name("rack").contents(lambda a: a.insert(lambda e: e.c2(p.instance["logger"]))))
problems = check(p)
assert problems == [
    "Structure.Container 'acquisition' contains Interactions.Instance 'other.spare' softly, but its hard container, "
    "Interactions.Implementation 'reference', does not contain it",
    "Interactions.Instance 'reference.logger' has 2 hard containers, Structure.Container 'rack', "
    "Interactions.Implementation 'reference': an object has at most one",
], problems
assert Validators.Validate(store).Reachable(p.schemas.Module, p.pipeline) == []

## CHK-05 · A schema refining an Interactions schema is checked, and named, as its own

In [ ]:
store, p = pipeline()
I = p.schemas
with store.transaction():
    Board = store.Schemas.OfObject(lambda o: o.name("acme.Board").ref(True).refines([I.Module]).properties([
        lambda t: t.name("revision").schema([store.Schemas.OfInt])]))
    Lane = store.Schemas.OfObject(lambda o: o.name("acme.Lane").ref(True).refines([I.Channel]))
with store.transaction():
    board = store.acme.Board(lambda b: b.revision(2).library(lambda a: a.insert(lambda e: e.name("Board").library(
        store.singleton("Interactions.Library")))))
    impl = store.Interactions.Implementation(lambda x: x.name("board_v2").module(lambda a: a.insert(lambda e: e.module(board))))
    lane = store.acme.Lane(lambda c: c.implementation(lambda a: a.insert(lambda e: e.name("lane0").impl(impl))))
assert Interactions.module_of(impl) is board and Interactions.path(lane) == "board_v2.lane0"
assert Interactions.Check()(board) == [
    "acme.Lane 'board_v2.lane0' carries messages of no schema",
    "acme.Lane 'board_v2.lane0' has no source",
    "acme.Lane 'board_v2.lane0' has no destination",
]